In [1]:
import chromadb
chroma_client = chromadb.Client()

In [2]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    device="cuda",
    normalize_embeddings=True   # unit vector
)

print("Sentence transformer hazir!")

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13488.61it/s]


Sentence transformer hazir!


In [4]:
collection = chroma_client.get_or_create_collection(
    name="IR_System_Collection",
    embedding_function=sentence_transformer_ef
)

In [31]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
print("Data set yüklendi")

Data set yüklendi


In [6]:
doc_texts = []
doc_ids = []

for doc in dataset.docs_iter():
    doc_texts.append(doc.text)
    doc_ids.append(doc.doc_id)

### Vector Database Storage

In [10]:
from tqdm import tqdm

# ChromaDB'nin maksimum batch limitini (5461) aşmamak için 5000 seçiyoruz
BATCH_SIZE = 5000 
toplam_veri_sayisi = len(doc_texts)

print(f"Toplam {toplam_veri_sayisi} doküman, {BATCH_SIZE}'lik paketler halinde ekleniyor...")

chroma_client.delete_collection("IR_System_Collection")     # Hata alınan eski koleksiyon silindi
collection = chroma_client.create_collection("IR_System_Collection", embedding_function=sentence_transformer_ef)

for i in tqdm(range(0, toplam_veri_sayisi, BATCH_SIZE)):
    
    # Listeleri 5000'lik parçalara bölüyoruz
    batch_texts = doc_texts[i : i + BATCH_SIZE]
    batch_ids = doc_ids[i : i + BATCH_SIZE]
    
    collection.add(
        documents=batch_texts,
        ids=batch_ids
    )

print("\nTüm veriler başarıyla vektörleştirildi ve kaydedildi!")

Toplam 369721 doküman, 5000'lik paketler halinde ekleniyor...


100%|██████████| 74/74 [1:16:24<00:00, 61.95s/it]


Tüm veriler başarıyla vektörleştirildi ve kaydedildi!


In [71]:
queries = []

for query in dataset.queries_iter():
    queries.append(query.text)

### Top 10

In [13]:
results = collection.query(
    query_texts=queries,
    n_results=10
)

print("En iyi 10 doküman!")

En iyi 10 doküman!


In [21]:
qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

In [22]:
def recall(found_list: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for doc_id in found_list:
        if doc_id in test_set:
            counter += 1

    return (counter / len(test_list)) * 100

In [23]:
def precision(found_list: list, test_list: list) -> float:
    if len(found_list) == 0:
        return 0.0

    counter = 0

    test_set = set(test_list)

    for doc_id in found_list:
        if doc_id in test_set:
            counter += 1

    return (counter / len(found_list)) * 100

In [24]:
import gc
del queries
del doc_texts
del doc_ids
gc.collect()

30

In [26]:
del dataset
gc.collect()

150

In [32]:
query_ids = [query.query_id for query in dataset.queries_iter()]

In [34]:
del dataset
gc.collect()

0

In [35]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [47]:
recall_10_list, precison_10_list = [], []

for idx in range(len(query_ids)):
    query_id = query_ids[idx]
    found_list = results['ids'][idx]
    test_list = qrels_dict[query_id]

    recall_10_list.append(recall(found_list, test_list))
    precison_10_list.append(precision(found_list, test_list))

In [50]:
def precision_AP(found_docs: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for doc in found_docs:
        if doc in test_set:
            counter += 1

    return counter / len(found_docs)


def AP(found_docs: list, test_list: list, value: int) -> float:
    total = 0

    for i in range(1, value + 1):
        if i < len(found_docs):
            precision_k = precision_AP(found_docs[:i], test_list)
            total += precision_k * (found_docs[i - 1] in set(test_list))

    return total / len(test_list)

In [51]:
AP_10_list = []

for idx in range(len(query_ids)):
    query_id = query_ids[idx]
    found_list = results['ids'][idx]
    test_list = qrels_dict[query_id]

    AP_10_list.append(AP(found_list, test_list, 10))

In [53]:
len(AP_10_list)

1444

In [54]:
import pandas as pd

df_10 = pd.DataFrame({
    "Query_ID": query_ids,
    "recall_10": recall_10_list,
    "precision_10": precison_10_list,
    "AP_10": AP_10_list
})

df_10

,Query_ID,recall_10,precision_10,AP_10
0,123839,33.333333,20.0,0.333333
1,188629,16.666667,10.0,0.166667
2,13898,16.666667,10.0,0.166667
3,316959,22.222222,20.0,0.129630
4,515031,7.142857,10.0,0.035714
...,...,...,...,...
1439,896124,25.000000,20.0,0.060714
1440,12319,4.545455,10.0,0.022727
1441,4421,6.666667,10.0,0.016667
1442,296526,0.000000,0.0,0.000000


In [55]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [56]:
results['distances']

[[0.4268239736557007,
  0.5009554624557495,
  0.5429425239562988,
  0.5572776794433594,
  0.5657732486724854,
  0.5676313638687134,
  0.5693140625953674,
  0.5703406929969788,
  0.5746500492095947,
  0.5792882442474365],
 [0.3844454288482666,
  0.4175373315811157,
  0.47837793827056885,
  0.481809139251709,
  0.4850879907608032,
  0.48882514238357544,
  0.48890072107315063,
  0.49097347259521484,
  0.49315208196640015,
  0.4977993369102478],
 [0.2145460844039917,
  0.35361188650131226,
  0.3784744143486023,
  0.39791548252105713,
  0.41265302896499634,
  0.416978120803833,
  0.4171130657196045,
  0.41884851455688477,
  0.42180335521698,
  0.42187267541885376],
 [0.37485504150390625,
  0.4067419171333313,
  0.4177553653717041,
  0.4263297915458679,
  0.4300188422203064,
  0.43280458450317383,
  0.4351937770843506,
  0.4448416233062744,
  0.45565325021743774,
  0.45706331729888916],
 [0.3709142208099365,
  0.39169609546661377,
  0.4057389497756958,
  0.4109640121459961,
  0.4167298078536

In [57]:
scores = []

for score_list in results['distances']:
    sub_scores = []

    for score in score_list:
        sub_scores.append(1 - score)

    scores.append(sub_scores)

In [59]:
from collections import defaultdict
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

class Scoredoc:
    def __init__(self, doc_id, score):
        self.doc_id = doc_id
        self.score = score

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

In [63]:
most_related_10_dict_with_scores = defaultdict(list)

for idx in range(len(query_ids)):
    for i in range(10):
        doc_id = results['ids'][idx][i]
        score = scores[idx][i]

        most_related_10_dict_with_scores[query_ids[idx]].append((doc_id, score))

In [65]:
from collections import defaultdict
from sklearn.metrics import ndcg_score
import numpy as np

doc_id_score_dict = defaultdict(float)
ndcg_10_list = []

for idx in range(len(df_10)):
    query_id = df_10.loc[idx, "Query_ID"]
    scoredoc_object_list = score_doc_dict[query_id]

    for scoreddoc_object in scoredoc_object_list:
        doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

    model_score_tuple_list = most_related_10_dict_with_scores[query_id]

    y_score, y_true = [], []

    for mytuple in model_score_tuple_list:
        doc_id = mytuple[0]
        score = mytuple[1]

        y_score.append(score)
        y_true.append(doc_id_score_dict[doc_id])

    if len(y_score) == 0:
        ndcg_10_list.append(0.0)
        continue

    if len(y_score) == 1:
        y_true.append(0.0)
        y_score.append(0.0)

    if len(y_true) == len(y_score):
        ndcg_10_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=10))
    else:
        print("There is a problem with query", query_id)

In [67]:
len(ndcg_10_list)

1444

In [68]:
for value in ndcg_10_list:
    try:
        myvalue = int(value)
    except:
        print("Problem")

In [69]:
df_10["NDCG_10"] = ndcg_10_list
df_10

,Query_ID,recall_10,precision_10,AP_10,NDCG_10
0,123839,33.333333,20.0,0.333333,0.918645
1,188629,16.666667,10.0,0.166667,0.847578
2,13898,16.666667,10.0,0.166667,0.000000
3,316959,22.222222,20.0,0.129630,0.693762
4,515031,7.142857,10.0,0.035714,0.859700
...,...,...,...,...,...
1439,896124,25.000000,20.0,0.060714,0.900632
1440,12319,4.545455,10.0,0.022727,0.922079
1441,4421,6.666667,10.0,0.016667,0.596932
1442,296526,0.000000,0.0,0.000000,0.753443


In [70]:
df_10.isna().sum()

Query_ID        0
recall_10       0
precision_10    0
AP_10           0
NDCG_10         0
dtype: int64

### Top 5

In [72]:
results_5 = collection.query(
    query_texts=queries,
    n_results=5
)

print("En iyi 5 doküman!")

En iyi 5 doküman!


In [73]:
recall_5_list, precison_5_list = [], []

for idx in range(len(query_ids)):
    query_id = query_ids[idx]
    found_list = results_5['ids'][idx]
    test_list = qrels_dict[query_id]

    recall_5_list.append(recall(found_list, test_list))
    precison_5_list.append(precision(found_list, test_list))

In [76]:
AP_5_list = []

for idx in range(len(query_ids)):
    query_id = query_ids[idx]
    found_list = results_5['ids'][idx]
    test_list = qrels_dict[query_id]

    AP_5_list.append(AP(found_list, test_list, 5))

In [79]:
scores_5 = []

for score_list in results_5['distances']:
    sub_scores_5 = []

    for score in score_list:
        sub_scores_5.append(1 - score)

    scores_5.append(sub_scores)

In [80]:
most_related_5_dict_with_scores = defaultdict(list)

for idx in range(len(query_ids)):
    for i in range(5):
        doc_id = results_5['ids'][idx][i]
        score = scores_5[idx][i]

        most_related_5_dict_with_scores[query_ids[idx]].append((doc_id, score))

In [81]:
from collections import defaultdict
from sklearn.metrics import ndcg_score
import numpy as np

doc_id_score_dict = defaultdict(float)
ndcg_5_list = []

for idx in range(len(df_10)):
    query_id = df_10.loc[idx, "Query_ID"]
    scoredoc_object_list = score_doc_dict[query_id]

    for scoreddoc_object in scoredoc_object_list:
        doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

    model_score_tuple_list = most_related_5_dict_with_scores[query_id]

    y_score, y_true = [], []

    for mytuple in model_score_tuple_list:
        doc_id = mytuple[0]
        score = mytuple[1]

        y_score.append(score)
        y_true.append(doc_id_score_dict[doc_id])

    if len(y_score) == 0:
        ndcg_5_list.append(0.0)
        continue

    if len(y_score) == 1:
        y_true.append(0.0)
        y_score.append(0.0)

    if len(y_true) == len(y_score):
        ndcg_5_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=5))
    else:
        print("There is a problem with query", query_id)

In [84]:
df_10["recall_5"] = recall_5_list
df_10["precision_5"] = precison_5_list
df_10["AP_5"] = AP_5_list
df_10["NDCG_5"] = ndcg_5_list

df_10["f_score_5"] = 2 * df_10["recall_5"] * df_10["precision_5"] / (df_10["recall_5"] + df_10["precision_5"])
df_10["f_score_10"] = 2 * df_10["recall_10"] * df_10["precision_10"] / (df_10["recall_10"] + df_10["precision_10"])

df_10

,Query_ID,recall_10,precision_10,AP_10,NDCG_10,recall_5,precision_5,AP_5,NDCG_5,f_score_5,f_score_10
0,123839,33.333333,20.0,0.333333,0.918645,33.333333,40.0,0.333333,1.000000,36.363636,25.000000
1,188629,16.666667,10.0,0.166667,0.847578,16.666667,20.0,0.166667,0.946333,18.181818,12.500000
2,13898,16.666667,10.0,0.166667,0.000000,16.666667,20.0,0.166667,0.000000,18.181818,12.500000
3,316959,22.222222,20.0,0.129630,0.693762,22.222222,40.0,0.129630,0.684296,28.571429,21.052632
4,515031,7.142857,10.0,0.035714,0.859700,7.142857,20.0,0.035714,0.898585,10.526316,8.333333
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,25.000000,20.0,0.060714,0.900632,12.500000,20.0,0.000000,0.937155,15.384615,22.222222
1440,12319,4.545455,10.0,0.022727,0.922079,4.545455,20.0,0.022727,0.980419,7.407407,6.250000
1441,4421,6.666667,10.0,0.016667,0.596932,6.666667,20.0,0.016667,0.540791,10.000000,8.000000
1442,296526,0.000000,0.0,0.000000,0.753443,0.000000,0.0,0.000000,0.693954,NaN,NaN


In [85]:
mydict = {
    "Method": "Sentence Transformer",
    "recall_5_mean": df_10["recall_5"].mean(),
    "recall_5_std": df_10["recall_5"].std(),
    "recall_5_max": df_10["recall_5"].max(),
    "recall_5_min": df_10["recall_5"].min(),
    "recall_10_mean": df_10["recall_10"].mean(),
    "recall_10_std": df_10["recall_10"].std(),
    "recall_10_max": df_10["recall_10"].max(),
    "recall_10_min": df_10["recall_10"].min(),
    "precision_5_mean": df_10["precision_5"].mean(),
    "precision_5_std": df_10["precision_5"].std(),
    "precision_5_max": df_10["precision_5"].max(),
    "precision_5_min": df_10["precision_5"].min(),
    "precision_10_mean": df_10["precision_10"].mean(),
    "precision_10_std": df_10["precision_10"].std(),
    "precision_10_max": df_10["precision_10"].max(),
    "precision_10_min": df_10["precision_10"].min(),
    "f_score_5_mean": df_10["f_score_5"].mean(),
    "f_score_5_std": df_10["f_score_5"].std(),
    "f_score_5_max": df_10["f_score_5"].max(),
    "f_score_5_min": df_10["f_score_5"].min(),
    "f_score_10_mean": df_10["f_score_10"].mean(),
    "f_score_10_std": df_10["f_score_10"].std(),
    "f_score_10_max": df_10["f_score_10"].max(),
    "f_score_10_min": df_10["f_score_10"].min(),
    "MAP_5": df_10["AP_5"].mean(),
    "MAP_10": df_10["AP_10"].mean(),
    "NDCG_5_mean": df_10["NDCG_5"].mean(),
    "NDCG_5_std": df_10["NDCG_5"].std(),
    "NDCG_5_max": df_10["NDCG_5"].max(),
    "NDCG_5_min": df_10["NDCG_5"].min(),
    "NDCG_10_mean": df_10["NDCG_10"].mean(),
    "NDCG_10_std": df_10["NDCG_10"].std(),
    "NDCG_10_max": df_10["NDCG_10"].max(),
    "NDCG_10_min": df_10["NDCG_10"].min()
}

In [86]:
df_parquet = pd.DataFrame(mydict, index=[0])
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,Sentence Transformer,12.190058,12.517122,66.666667,0.0,15.646733,15.715276,85.714286,0.0,27.202216,...,0.094242,0.112406,0.6981,0.363412,1.0,0.0,0.70553,0.307505,1.0,0.0


In [87]:
df_parquet.to_parquet("SentenceTransformer.parquet")